# Aing_리그전 Transformer Fine-tune
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aing-gachon/26-Spring-Transformer-Study/blob/main/Week3/Aing_%E1%84%85%E1%85%B5%E1%84%80%E1%85%B3%E1%84%8C%E1%85%A5%E1%86%AB_Transformer_Finetune.ipynb)

이 노트북은 **고정 pre-train 모델을 한 번 만든 뒤, fine-tuning 하이퍼파라미터만 바꿔 보는 리그전 코드**입니다.

- Stage A: Multi30k로 고정 Transformer를 한 번 pre-train합니다.
- Stage B: IWSLT로 fine-tuning합니다.
- 참가자는 마지막의 `LEAGUE_MODE + FT_HP` 셀과 바로 아래 fine-tuning 실행 셀만 반복 실행합니다.
- 최종 점수는 `LEAGUE_MODE = "final"`에서 출력되는 `LEAGUE_FINAL_SCORE_IWSLT_TEST_BLEU`입니다.

---


### STEP1. 설치

이 STEP은 Colab에서 필요한 라이브러리를 설치합니다.

- `transformers`: Transformer/BART 모델과 `Seq2SeqTrainer` 사용
- `datasets`: Multi30k, IWSLT 데이터 로드
- `tokenizers`: BPE tokenizer 학습
- `sacrebleu`: 번역 성능 BLEU 계산

런타임을 새로 시작했다면 이 STEP부터 다시 실행합니다.


In [ ]:
!pip -q install -U "transformers==5.2.0" "datasets==4.5.0" "accelerate==1.12.0" "tokenizers==0.22.2" sacrebleu

# 설치 후 import 에러가 나면 런타임 재시작(Runtime > Restart runtime) 후 다시 실행하세요.

### STEP2. 라이브러리 import

이 STEP은 이후 코드에서 사용할 기본 라이브러리와 HuggingFace 관련 도구를 불러옵니다.

- 데이터 처리: `datasets`, `numpy`
- 모델 학습: `torch`, `transformers`
- 토크나이저: `tokenizers`, `PreTrainedTokenizerFast`
- 평가: `sacrebleu`

이 STEP은 코드를 실행하기 위한 준비 단계이므로 값을 수정하지 않습니다.


In [ ]:
import os, math, random, time
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # tokenizer 병렬 경고/데드락 방지
from dataclasses import dataclass
from typing import Dict, Any, Optional

import numpy as np
import torch
import sacrebleu

from datasets import load_dataset, Dataset

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing

from transformers import (
    PreTrainedTokenizerFast,
    BartConfig,
    BartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

def safe_load_dataset(*args, **kwargs):
    try:
        return load_dataset(*args, **kwargs)
    except Exception as e:
        if 'trust_remote_code' in str(e):
            return load_dataset(*args, trust_remote_code=True, **kwargs)
        raise

# === 공통 유틸: BPE 토크나이저 학습 / HF 토크나이저 래퍼 ===
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>"]

def train_bpe_tokenizer(train_ds, vocab_size=8000, min_freq=2, src_key=None, tgt_key=None):
    if src_key is None:
        src_key = globals().get("SRC_KEY", "de")
    if tgt_key is None:
        tgt_key = globals().get("TGT_KEY", "en")

    tok = Tokenizer(BPE(unk_token="<unk>"))
    tok.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=min_freq,
        special_tokens=SPECIAL_TOKENS,
    )

    def iterator():
        for ex in train_ds:
            yield ex[src_key]
            yield ex[tgt_key]

    tok.train_from_iterator(iterator(), trainer=trainer)

    bos_id = tok.token_to_id("<bos>")
    eos_id = tok.token_to_id("<eos>")
    tok.post_processor = TemplateProcessing(
        single="<bos> $A <eos>",
        special_tokens=[("<bos>", bos_id), ("<eos>", eos_id)],
    )
    return tok

def build_hf_tokenizer_from_tokenizers(tok: Tokenizer) -> PreTrainedTokenizerFast:
    return PreTrainedTokenizerFast(
        tokenizer_object=tok,
        bos_token="<bos>",
        eos_token="<eos>",
        unk_token="<unk>",
        pad_token="<pad>",
    )


### STEP3. seed / device 설정

이 STEP은 실험의 재현성을 위해 random seed를 고정하고, GPU 사용 가능 여부를 확인합니다.

- seed를 고정하면 실행할 때마다 결과가 크게 달라지는 현상을 줄일 수 있습니다.
- `cuda`가 잡히면 GPU로 학습합니다.

리그전 중에는 이 STEP을 수정하지 않습니다.


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

def make_seq2seq_training_args(**kwargs):
    try:
        return Seq2SeqTrainingArguments(**kwargs)
    except TypeError:
        # 가장 흔한 변경: eval_strategy ↔ evaluation_strategy
        if "eval_strategy" in kwargs and "evaluation_strategy" not in kwargs:
            kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
        elif "evaluation_strategy" in kwargs and "eval_strategy" not in kwargs:
            kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
        return Seq2SeqTrainingArguments(**kwargs)


### STEP3-1. 리그전 공통 고정 설정

이 STEP은 리그전에서 바꾸지 않는 고정값을 정의합니다.

- 고정 Transformer 모델 구조
- tokenizer / 데이터 / pre-train 설정
- `public_fast`, `public_full`, `final` 모드별 fine-tuning 설정
- fine-tuning 실행 셀 기준 예상 시간

참가자가 수정하는 값은 이 STEP이 아니라 마지막 `STEP12-A`의 `LEAGUE_MODE`와 `FT_HP`입니다.


In [ ]:
# === League fixed global settings ===

PRETRAIN_MODEL_TAG = "fixed_small_transformer"
MODEL_HP = dict(
    d_model=256,
    nhead=4,
    num_layers=3,
    d_ff=1024,
    dropout=0.10,
)

M30K_TRAIN_SAMPLES_FIXED = 29_000
TOKENIZER_IWSLT_SAMPLES_FIXED = 80_000
IWSLT_SHUFFLE_SEED_FIXED = 42
VOCAB_SIZE_FIXED = 12_000
MIN_FREQ_FIXED = 2
MAX_LEN_FIXED = 80

PRETRAIN_MAX_STEPS_FIXED = 1_000
PRETRAIN_LR_FIXED = 5e-4
PRETRAIN_WARMUP_FIXED = 200
PRETRAIN_SCHEDULER_FIXED = "inverse_sqrt"
PRETRAIN_EVAL_STEPS_FIXED = 250
PRETRAIN_BSZ_FIXED = 8
PRETRAIN_ACCUM_FIXED = 4
PRETRAIN_GEN_BEAMS_FIXED = 1

MODE_CONFIGS = {
    "public_fast": dict(
        iwslt_train_samples=12_000,
        finetune_max_steps=300,
        finetune_eval_steps=100,
        valid_eval_samples=512,
        gen_beams=1,
        run_test_eval=False,
        expected_minutes="약 3분",
    ),
    "public_full": dict(
        iwslt_train_samples=80_000,
        finetune_max_steps=1_000,
        finetune_eval_steps=200,
        valid_eval_samples=None,
        gen_beams=1,
        run_test_eval=False,
        expected_minutes="약 8분",
    ),
    "final": dict(
        iwslt_train_samples=80_000,
        finetune_max_steps=1_000,
        finetune_eval_steps=200,
        valid_eval_samples=None,
        gen_beams=1,
        run_test_eval=True,
        expected_minutes="약 20분",
    ),
}

FINETUNE_BSZ_FIXED = 8
FINETUNE_ACCUM_FIXED = 4

def get_mode_config(league_mode):
    if league_mode not in MODE_CONFIGS:
        raise ValueError(f"LEAGUE_MODE must be one of {list(MODE_CONFIGS)}")
    return MODE_CONFIGS[league_mode].copy()

print("Fixed MODEL_HP:", MODEL_HP)
print("Available LEAGUE_MODE:", list(MODE_CONFIGS))
print("Estimated rerun time after Stage A: public_fast≈3분, public_full≈8분, final≈20분")


## 1) 데이터 로드 & 토크나이저 준비

이 구간에서는 번역 학습에 사용할 데이터와 tokenizer를 준비합니다.

- Multi30k: Stage A pre-train에 사용
- IWSLT: Stage B fine-tuning과 validation/test 평가에 사용
- BPE tokenizer: 문장을 모델 입력 token id로 바꾸는 역할

여기까지는 실험을 위한 준비 과정입니다.


### STEP4. Multi30k/IWSLT 데이터 로드

이 STEP은 두 데이터셋을 불러옵니다.

- Multi30k: 기본 독일어→영어 번역 능력을 만들기 위한 pre-train 데이터
- IWSLT: fine-tuning과 최종 평가에 사용하는 데이터

참가자는 이 STEP을 수정하지 않습니다.


In [ ]:
multi30k = safe_load_dataset("bentrevett/multi30k")
SRC_KEY = "de"
TGT_KEY = "en"

train_m30k = multi30k["train"]
valid_m30k = multi30k["validation"]
test_m30k  = multi30k["test"]

# === (고정) Multi30k 데이터 사용 규칙 ===
train_m30k = train_m30k.select(range(min(M30K_TRAIN_SAMPLES_FIXED, len(train_m30k))))

print("Multi30k sizes:", len(train_m30k), len(valid_m30k), len(test_m30k))
print("Multi30k sample:", train_m30k[0])

# === (고정) IWSLT 데이터 로드 ===
def load_iwslt2017_de_en_parquet():
    """IWSLT 2017 (de-en) parquet 직접 로드."""
    REV = "b2dcd74c28e4544eb5c9bb76ace449ea26bbe909"
    base = f"https://huggingface.co/datasets/IWSLT/iwslt2017/resolve/{REV}/iwslt2017-de-en"

    data_files = {
        "train": f"{base}/iwslt2017-train.parquet",
        "validation": f"{base}/iwslt2017-validation.parquet",
        "test": f"{base}/iwslt2017-test.parquet",
    }
    return load_dataset("parquet", data_files=data_files)

ft_raw = load_iwslt2017_de_en_parquet()
print("IWSLT sizes:", len(ft_raw["train"]), len(ft_raw["validation"]), len(ft_raw["test"]))
print("IWSLT sample:", ft_raw["train"][0])


### STEP5. BPE tokenizer 준비

이 STEP은 문장을 token id로 바꾸는 tokenizer를 준비합니다.

- BPE tokenizer는 단어를 더 작은 subword 단위로 나눕니다.
- Multi30k와 IWSLT 문장을 함께 사용해 tokenizer를 학습합니다.
- tokenizer 설정은 리그전에서 고정입니다.

마지막 fine-tuning 셀만 다시 실행할 때는 이 STEP을 다시 실행하지 않습니다.


In [ ]:
assert "build_hf_tokenizer_from_tokenizers" in globals(), "build_hf_tokenizer_from_tokenizers가 없습니다. 위의 import/유틸(토크나이저) 셀을 먼저 실행하세요."
assert "ft_raw" in globals(), "IWSLT 원본 데이터(ft_raw)가 없습니다. STEP4 셀을 먼저 실행하세요."

import json

# === IWSLT translation field helper ===
def extract_translation_pair(trans, src_lang="de", tgt_lang="en"):
    """IWSLT translation 필드(dict 또는 JSON string)에서 src/tgt 문장을 꺼냅니다."""
    if isinstance(trans, str):
        try:
            trans = json.loads(trans)
        except Exception:
            trans = {}

    if isinstance(trans, dict):
        return trans.get(src_lang, ""), trans.get(tgt_lang, "")
    return "", ""

# === BPE tokenizer helper ===
def train_bpe_tokenizer_from_text_iterator(text_iterator_fn, vocab_size=12_000, min_freq=2):
    tok = Tokenizer(BPE(unk_token="<unk>"))
    tok.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=min_freq,
        special_tokens=SPECIAL_TOKENS,
    )

    tok.train_from_iterator(text_iterator_fn(), trainer=trainer)

    bos_id = tok.token_to_id("<bos>")
    eos_id = tok.token_to_id("<eos>")
    tok.post_processor = TemplateProcessing(
        single="<bos> $A <eos>",
        special_tokens=[("<bos>", bos_id), ("<eos>", eos_id)],
    )
    return tok

# IWSLT subset은 tokenizer 학습과 fine-tune에서 같은 seed를 사용합니다.
iwslt_tok_train = ft_raw["train"].shuffle(seed=IWSLT_SHUFFLE_SEED_FIXED).select(
    range(min(TOKENIZER_IWSLT_SAMPLES_FIXED, len(ft_raw["train"])))
)

def tokenizer_text_iterator():
    # Multi30k text
    for ex in train_m30k:
        yield ex[SRC_KEY]
        yield ex[TGT_KEY]

    # IWSLT text
    for ex in iwslt_tok_train:
        src_text, tgt_text = extract_translation_pair(ex["translation"], "de", "en")
        yield src_text
        yield tgt_text

tok = train_bpe_tokenizer_from_text_iterator(
    tokenizer_text_iterator,
    vocab_size=VOCAB_SIZE_FIXED,
    min_freq=MIN_FREQ_FIXED,
)
hf_tokenizer = build_hf_tokenizer_from_tokenizers(tok)

MAX_LEN = MAX_LEN_FIXED

print("Trained tokenizer in current runtime")
print("vocab_size:", hf_tokenizer.vocab_size)
print("MAX_LEN:", MAX_LEN)
print("special tokens:", hf_tokenizer.special_tokens_map)


### STEP6. 공통 전처리 함수

이 STEP은 문장 데이터를 모델 학습용 입력으로 바꾸는 함수를 정의합니다.

- `input_ids`: 독일어 문장을 tokenizer로 바꾼 입력 token id
- `labels`: 영어 정답 문장을 tokenizer로 바꾼 정답 token id
- `MAX_LEN`: 너무 긴 문장을 자르는 최대 길이

함수 정의 단계이므로 한 번만 실행하면 됩니다.


In [ ]:
# === 공통 전처리 함수 ===
def preprocess_m30k(examples):
    model_inputs = hf_tokenizer(examples[SRC_KEY], max_length=MAX_LEN, truncation=True)
    labels = hf_tokenizer(text_target=examples[TGT_KEY], max_length=MAX_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_m30k_tok = train_m30k.map(preprocess_m30k, batched=True, remove_columns=train_m30k.column_names)
valid_m30k_tok = valid_m30k.map(preprocess_m30k, batched=True, remove_columns=valid_m30k.column_names)
test_m30k_tok  = test_m30k.map(preprocess_m30k,  batched=True, remove_columns=test_m30k.column_names)

train_m30k_tok[0]

### STEP7. 고정 Transformer 모델 생성

이 STEP은 리그전에서 사용할 고정 Transformer 모델을 만듭니다.

- 모델 구조(`MODEL_HP`)는 고정입니다.
- 참가자는 모델 크기나 layer 수를 바꾸지 않습니다.
- 이후 Stage A에서 이 모델을 pre-train합니다.


In [ ]:
# === Fixed model 생성 ===

print("Fixed MODEL_HP:", MODEL_HP)

def build_model(d_model, nhead, num_layers, d_ff, dropout):
    assert d_model % nhead == 0, f"d_model({d_model}) must be divisible by nhead({nhead})"

    config = BartConfig(
        vocab_size=hf_tokenizer.vocab_size,
        d_model=d_model,
        encoder_layers=num_layers,
        decoder_layers=num_layers,
        encoder_attention_heads=nhead,
        decoder_attention_heads=nhead,
        encoder_ffn_dim=d_ff,
        decoder_ffn_dim=d_ff,
        dropout=dropout,
        attention_dropout=dropout,
        activation_dropout=dropout,
        max_position_embeddings=MAX_LEN + 2,
        pad_token_id=hf_tokenizer.pad_token_id,
        bos_token_id=hf_tokenizer.bos_token_id,
        eos_token_id=hf_tokenizer.eos_token_id,
        decoder_start_token_id=hf_tokenizer.bos_token_id,
        attn_implementation="eager",
    )
    return BartForConditionalGeneration(config)

model = build_model(**MODEL_HP).to(device)

if model.config.vocab_size != hf_tokenizer.vocab_size:
    raise ValueError(f"model vocab_size({model.config.vocab_size}) != tokenizer vocab_size({hf_tokenizer.vocab_size})")

print("model params (M):", sum(p.numel() for p in model.parameters()) / 1e6)


### STEP8. BLEU 평가 함수 정의

이 STEP은 번역 결과를 BLEU 점수로 계산하는 함수를 정의합니다.

- BLEU는 번역 결과가 정답 번역과 얼마나 비슷한지 보는 지표입니다.
- 높을수록 좋은 점수입니다.
- 리그전 점수는 `final` 모드의 IWSLT test BLEU입니다.

평가 함수 정의 단계이므로 값을 수정하지 않습니다.


In [ ]:
import re

_detok_punct_re = re.compile(r"\s+([?.!,;:])")
_detok_contraction_re = re.compile(r"\s+(n't|'re|'ve|'ll|'d|'m|'s)\b")

def simple_detokenize(text: str) -> str:
    text = text.strip()
    text = _detok_punct_re.sub(r"\1", text)
    text = _detok_contraction_re.sub(r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def compute_metrics(eval_pred):
    preds = eval_pred.predictions if hasattr(eval_pred, "predictions") else eval_pred[0]
    labels = eval_pred.label_ids if hasattr(eval_pred, "label_ids") else eval_pred[1]

    if isinstance(preds, tuple):
        preds = preds[0]

    pad_id = hf_tokenizer.pad_token_id
    unk_id = hf_tokenizer.unk_token_id if hf_tokenizer.unk_token_id is not None else pad_id

    preds  = np.where(preds  != -100, preds,  pad_id)
    labels = np.where(labels != -100, labels, pad_id)

    preds  = np.where((preds  < 0) | (preds  >= hf_tokenizer.vocab_size), unk_id, preds)
    labels = np.where((labels < 0) | (labels >= hf_tokenizer.vocab_size), unk_id, labels)

    pred_str  = hf_tokenizer.batch_decode(preds,  skip_special_tokens=True)
    label_str = hf_tokenizer.batch_decode(labels, skip_special_tokens=True)

    pred_str  = [simple_detokenize(p) for p in pred_str]
    label_str = [simple_detokenize(l) for l in label_str]

    bleu = sacrebleu.corpus_bleu(pred_str, [label_str], force=True).score
    return {"bleu": bleu}


### STEP9. Stage A: Fixed Pre-train

이 STEP은 Multi30k로 고정 모델을 한 번 pre-train합니다.

- 리그전 시작 전에 한 번만 실행합니다.
- 이후 `LEAGUE_MODE`나 `FT_HP`를 바꿀 때는 이 STEP을 다시 실행하지 않습니다.
- 반복 실험은 `STEP12-A`와 `STEP12-B`에서 진행합니다.


In [ ]:
# === Stage A: Fixed Pre-train on Multi30k ===
from datetime import datetime
import copy, gc

PRETRAIN_RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
PRETRAIN_OUT_DIR = f"./runs/league_pretrain_multi30k_fixed_{PRETRAIN_RUN_TAG}"

pre_args = make_seq2seq_training_args(
    output_dir=PRETRAIN_OUT_DIR,
    max_steps=PRETRAIN_MAX_STEPS_FIXED,
    logging_steps=50,
    eval_steps=PRETRAIN_EVAL_STEPS_FIXED,
    eval_strategy="steps",

    save_strategy="no",
    load_best_model_at_end=False,

    per_device_train_batch_size=PRETRAIN_BSZ_FIXED,
    per_device_eval_batch_size=PRETRAIN_BSZ_FIXED,
    gradient_accumulation_steps=PRETRAIN_ACCUM_FIXED,

    learning_rate=PRETRAIN_LR_FIXED,
    lr_scheduler_type=PRETRAIN_SCHEDULER_FIXED,
    warmup_steps=PRETRAIN_WARMUP_FIXED,

    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    generation_num_beams=PRETRAIN_GEN_BEAMS_FIXED,

    fp16=torch.cuda.is_available(),
    report_to=[],
)

collator = DataCollatorForSeq2Seq(hf_tokenizer, model=model)

try:
    pre_trainer = Seq2SeqTrainer(
        model=model,
        args=pre_args,
        train_dataset=train_m30k_tok,
        eval_dataset=valid_m30k_tok,
        processing_class=hf_tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )
except TypeError:
    pre_trainer = Seq2SeqTrainer(
        model=model,
        args=pre_args,
        train_dataset=train_m30k_tok,
        eval_dataset=valid_m30k_tok,
        tokenizer=hf_tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )

pre_train_result = pre_trainer.train()
pre_valid = pre_trainer.evaluate()
print("pre_valid:", pre_valid)

pre_log_history = pre_trainer.state.log_history
model = pre_trainer.model
stage1_eval_after = pre_valid

BASE_MODEL_STATE = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
BASE_MODEL_READY = True
print("Stage A 완료. 이제 STEP12-A와 STEP12-B를 반복 실행할 수 있습니다.")

model.to("cpu")
del pre_trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


### STEP10. Fine-tune 데이터 준비 안내

이 STEP은 IWSLT 데이터가 모드별로 어떻게 준비되는지 확인하는 단계입니다.

- `public_fast`: 빠른 실험을 위해 train/validation 일부 사용
- `public_full`: 전체 validation 기준으로 후보 확인
- `final`: test BLEU까지 계산

실제 tokenization은 다음 STEP의 함수가 처리합니다.


In [ ]:
# === Fine-tune 데이터 준비 안내 ===
print("IWSLT raw train/validation/test sizes:", len(ft_raw["train"]), len(ft_raw["validation"]), len(ft_raw["test"]))
print("Available LEAGUE_MODE:", list(MODE_CONFIGS))
ft_raw


### STEP11. Fine-tune 전처리 함수

이 STEP은 IWSLT 데이터를 fine-tuning용으로 전처리하는 함수를 정의합니다.

- `LEAGUE_MODE`에 맞게 train/validation/test 데이터를 준비합니다.
- 한 번만 실행하면 됩니다.


In [ ]:
# === Fine-tune 전처리 함수 ===
def preprocess_translation_dict(examples, src_lang="de", tgt_lang="en"):
    trans = examples["translation"]

    src_texts, tgt_texts = [], []
    for ex in trans:
        src_text, tgt_text = extract_translation_pair(ex, src_lang, tgt_lang)
        src_texts.append(src_text)
        tgt_texts.append(tgt_text)

    model_inputs = hf_tokenizer(src_texts, max_length=MAX_LEN, truncation=True)

    try:
        labels = hf_tokenizer(text_target=tgt_texts, max_length=MAX_LEN, truncation=True)
    except TypeError:
        with hf_tokenizer.as_target_tokenizer():
            labels = hf_tokenizer(tgt_texts, max_length=MAX_LEN, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

FT_DATA_CACHE = {}

def get_ft_datasets_for_mode(league_mode):
    cfg = get_mode_config(league_mode)
    cache_key = league_mode

    if cache_key in FT_DATA_CACHE:
        return FT_DATA_CACHE[cache_key]

    print(f"Preparing IWSLT datasets for mode: {league_mode}")
    ft_train = ft_raw["train"].shuffle(seed=IWSLT_SHUFFLE_SEED_FIXED).select(
        range(min(cfg["iwslt_train_samples"], len(ft_raw["train"])))
    )
    ft_valid = ft_raw["validation"]
    ft_test = ft_raw["test"]

    ft_train_tok = ft_train.map(
        lambda x: preprocess_translation_dict(x, "de", "en"),
        batched=True,
        remove_columns=ft_train.column_names,
    )
    ft_valid_tok = ft_valid.map(
        lambda x: preprocess_translation_dict(x, "de", "en"),
        batched=True,
        remove_columns=ft_valid.column_names,
    )

    if cfg["valid_eval_samples"] is None:
        ft_valid_eval_tok = ft_valid_tok
    else:
        ft_valid_eval_tok = ft_valid_tok.select(range(min(cfg["valid_eval_samples"], len(ft_valid_tok))))

    if cfg["run_test_eval"]:
        ft_test_tok = ft_test.map(
            lambda x: preprocess_translation_dict(x, "de", "en"),
            batched=True,
            remove_columns=ft_test.column_names,
        )
    else:
        ft_test_tok = None

    data = dict(
        train=ft_train_tok,
        valid_full=ft_valid_tok,
        valid_eval=ft_valid_eval_tok,
        test=ft_test_tok,
        config=cfg,
    )
    FT_DATA_CACHE[cache_key] = data

    print("IWSLT train samples:", len(ft_train_tok))
    print("IWSLT valid eval samples:", len(ft_valid_eval_tok), "/", len(ft_valid_tok))
    print("IWSLT test eval enabled:", cfg["run_test_eval"])
    return data


### STEP12-A. 실험 설정 셀: `LEAGUE_MODE` + `FT_HP`

참가자가 실험할 때 주로 수정하는 셀입니다.

- `LEAGUE_MODE = "public_fast"`: 여러 조합을 빠르게 비교할 때 사용합니다. fine-tuning 실행 셀 재실행 기준 **약 3분**이 걸립니다.
- `LEAGUE_MODE = "public_full"`: 좋은 후보를 전체 validation으로 확인할 때 사용합니다. fine-tuning 실행 셀 재실행 기준 **약 8분**이 걸립니다.
- `LEAGUE_MODE = "final"`: 최종 test BLEU를 계산할 때 사용합니다. fine-tuning 실행 셀 재실행 기준 **약 20분**이 걸립니다.

이 셀에서 `LEAGUE_MODE`와 `FT_HP`만 수정한 뒤, 바로 아래 `STEP12-B` fine-tuning 실행 셀만 다시 실행합니다.


In [ ]:
# === 실험 설정 셀: LEAGUE_MODE + FT_HP ===

LEAGUE_MODE = "public_fast"  # "public_fast" / "public_full" / "final"

FT_HP = dict(
    learning_rate=1e-4,            # 🔧 TUNE: 1e-5 ~ 5e-4
    lr_scheduler_type="linear",    # 🔧 TUNE: linear / cosine / inverse_sqrt
    warmup_steps=50,               # 🔧 TUNE: 0 ~ 300
    weight_decay=0.0,              # 🔧 TUNE: 0.0 ~ 0.1
    label_smoothing_factor=0.10,   # 🔧 TUNE: 0.0 ~ 0.20
)

def validate_ft_hp(ft_hp):
    required_keys = {
        "learning_rate",
        "lr_scheduler_type",
        "warmup_steps",
        "weight_decay",
        "label_smoothing_factor",
    }
    if set(ft_hp.keys()) != required_keys:
        raise ValueError(f"FT_HP keys must be exactly {required_keys}")

    lr = float(ft_hp["learning_rate"])
    warmup = int(ft_hp["warmup_steps"])
    wd = float(ft_hp["weight_decay"])
    smoothing = float(ft_hp["label_smoothing_factor"])

    if not (1e-5 <= lr <= 5e-4):
        raise ValueError("learning_rate must be between 1e-5 and 5e-4")
    if ft_hp["lr_scheduler_type"] not in {"linear", "cosine", "inverse_sqrt"}:
        raise ValueError("lr_scheduler_type must be one of: linear, cosine, inverse_sqrt")
    if not (0 <= warmup <= 300):
        raise ValueError("warmup_steps must be between 0 and 300")
    if not (0.0 <= wd <= 0.1):
        raise ValueError("weight_decay must be between 0.0 and 0.1")
    if not (0.0 <= smoothing <= 0.2):
        raise ValueError("label_smoothing_factor must be between 0.0 and 0.2")

validate_ft_hp(FT_HP)
MODE_CONFIG = get_mode_config(LEAGUE_MODE)
RUN_TEST_EVAL = MODE_CONFIG["run_test_eval"]

print("LEAGUE_MODE:", LEAGUE_MODE)
print("FT_HP:", FT_HP)
print("MODE_CONFIG:", MODE_CONFIG)
print("RUN_TEST_EVAL:", RUN_TEST_EVAL)
print("Expected rerun time for fine-tuning cell:", MODE_CONFIG.get("expected_minutes"))


### STEP12-B. Fine-tune 실행 셀

이 STEP은 실제 fine-tuning을 수행하는 실행 셀입니다.

- `FT_HP`만 바꿨다면 `STEP12-A`와 이 셀만 다시 실행합니다.
- `LEAGUE_MODE`를 바꿨다면 `STEP12-A`와 이 셀만 다시 실행합니다.
- Stage A pre-train 셀은 다시 실행하지 않습니다.

예상 시간은 fine-tuning 실행 셀 재실행 기준으로 `public_fast` 약 3분, `public_full` 약 8분, `final` 약 20분입니다.


In [ ]:
# === Fine-tune execution cell ===

import gc
from datetime import datetime

assert "BASE_MODEL_READY" in globals() and BASE_MODEL_READY, (
    "STEP9 Stage A pre-train을 먼저 한 번 실행하세요."
)
validate_ft_hp(FT_HP)
MODE_CONFIG = get_mode_config(LEAGUE_MODE)
RUN_TEST_EVAL = MODE_CONFIG["run_test_eval"]

if "ft_trainer" in globals():
    try:
        del ft_trainer
    except Exception:
        pass
if "model" in globals():
    try:
        model.to("cpu")
    except Exception:
        pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

ft_data = get_ft_datasets_for_mode(LEAGUE_MODE)
ft_train_tok = ft_data["train"]
ft_valid_eval_tok = ft_data["valid_eval"]
ft_test_tok = ft_data["test"]

model = build_model(**MODEL_HP)
model.load_state_dict(BASE_MODEL_STATE, strict=True)
model.to(device)

FT_RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
FT_OUT_DIR = f"./runs/league_finetune_iwslt_{LEAGUE_MODE}_{PRETRAIN_MODEL_TAG}_{FT_RUN_TAG}"

ft_args = make_seq2seq_training_args(
    output_dir=FT_OUT_DIR,
    max_steps=MODE_CONFIG["finetune_max_steps"],
    logging_steps=50,
    eval_steps=MODE_CONFIG["finetune_eval_steps"],
    eval_strategy="steps",

    save_strategy="no",
    load_best_model_at_end=False,

    per_device_train_batch_size=FINETUNE_BSZ_FIXED,
    per_device_eval_batch_size=FINETUNE_BSZ_FIXED,
    gradient_accumulation_steps=FINETUNE_ACCUM_FIXED,

    learning_rate=FT_HP["learning_rate"],
    lr_scheduler_type=FT_HP["lr_scheduler_type"],
    warmup_steps=FT_HP["warmup_steps"],
    weight_decay=FT_HP["weight_decay"],
    label_smoothing_factor=FT_HP["label_smoothing_factor"],

    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    generation_num_beams=MODE_CONFIG["gen_beams"],

    fp16=torch.cuda.is_available(),
    report_to=[],
)

try:
    ft_trainer = Seq2SeqTrainer(
        model=model,
        args=ft_args,
        train_dataset=ft_train_tok,
        eval_dataset=ft_valid_eval_tok,
        processing_class=hf_tokenizer,
        data_collator=DataCollatorForSeq2Seq(hf_tokenizer, model=model),
        compute_metrics=compute_metrics,
    )
except TypeError:
    ft_trainer = Seq2SeqTrainer(
        model=model,
        args=ft_args,
        train_dataset=ft_train_tok,
        eval_dataset=ft_valid_eval_tok,
        tokenizer=hf_tokenizer,
        data_collator=DataCollatorForSeq2Seq(hf_tokenizer, model=model),
        compute_metrics=compute_metrics,
    )

ft_train_result = ft_trainer.train()

if LEAGUE_MODE == "final":
    ft_iwslt_valid = None
    VALID_BLEU = None
else:
    ft_iwslt_valid = ft_trainer.evaluate(eval_dataset=ft_valid_eval_tok, metric_key_prefix="iwslt_valid")
    VALID_BLEU = ft_iwslt_valid.get("iwslt_valid_bleu")

if RUN_TEST_EVAL:
    assert ft_test_tok is not None, "RUN_TEST_EVAL=True이면 ft_test_tok가 준비되어 있어야 합니다."
    ft_iwslt_test = ft_trainer.evaluate(eval_dataset=ft_test_tok, metric_key_prefix="iwslt_test")
    FINAL_TEST_BLEU = ft_iwslt_test.get("iwslt_test_bleu")
else:
    ft_iwslt_test = None
    FINAL_TEST_BLEU = None

print("Fine-tune train result:", ft_train_result)
if VALID_BLEU is not None:
    print("VALID_BLEU:", VALID_BLEU)
    print("LEAGUE_VALID_SCORE_IWSLT_VALID_BLEU:", VALID_BLEU)

if FINAL_TEST_BLEU is not None:
    print("FINAL_TEST_BLEU:", FINAL_TEST_BLEU)
    print("\n=== LEAGUE FINAL SCORE ===")
    print("LEAGUE_FINAL_SCORE_IWSLT_TEST_BLEU:", FINAL_TEST_BLEU)
else:
    print("\n=== VALIDATION SCORE ONLY ===")
    print("Set LEAGUE_MODE = 'final' to compute IWSLT test BLEU.")

run_summary = {
    "league_mode": LEAGUE_MODE,
    "ft_hp": FT_HP,
    "valid_bleu": VALID_BLEU,
    "final_iwslt_test_bleu": FINAL_TEST_BLEU,
    "fixed": {
        "model_hp": MODEL_HP,
        "max_len": MAX_LEN,
        "vocab_size": VOCAB_SIZE_FIXED,
        "iwslt_train_samples": MODE_CONFIG["iwslt_train_samples"],
        "finetune_max_steps": MODE_CONFIG["finetune_max_steps"],
        "gen_beams": MODE_CONFIG["gen_beams"],
        "expected_minutes": MODE_CONFIG.get("expected_minutes"),
    },
}

print("\n=== RUN SUMMARY (copy-paste) ===")
print(json.dumps(run_summary, ensure_ascii=False, indent=2))

ft_log_history = ft_trainer.state.log_history


---
## 📌 제작 정보 & 출처

- 제작: 가천대학교 인공지능 학술 동아리 **Aing (A.ing)**

### 사용/참고 자료
- Vaswani et al., **Attention Is All You Need**, NeurIPS 2017.
- A.ing 내부 스터디 자료: *Attention 치트시트*, *Transformer CookBook*.
- HuggingFace Transformers/Datasets 문서: Seq2Seq 학습(Trainer), Multi30k, IWSLT 로딩.
- `tokenizers`(BPE), `sacrebleu`(BLEU 평가) 라이브러리.
